# ScopeKeeper

**An agent that protects a freelancer's project from silently growing beyond what was agreed.**

CSE476 Agentic AI - Project 1

## 1. The problem

A freelancer agrees to a project with a clear scope - a set of deliverables, a budget, an hours estimate.
During the project, the client keeps asking for "just one more small thing." Each request, looked at on
its own, can seem reasonable. But added together, they can quietly turn the agreed project into a much
bigger one, without the freelancer ever being paid for the difference, and often without either side
noticing until it's too late.

This isn't a rare edge case. Industry write-ups put the cost of unpaid scope creep at **roughly $7,800-$15,600
per freelancer per year**, and there are already several commercial tools (ScopeShield, StopScopeCreep,
ScopeAuditor, ScopeGuard) charging $9-$20/month to help with this - which is itself evidence that people
consider the pain worth paying to fix.

## 2. What ScopeKeeper does

The freelancer describes the original deal once (deliverables, budget, hours, what's explicitly excluded).
From then on, whenever a new client request comes in, ScopeKeeper:

1. Reads the request and compares it against the agreed scope.
2. Decides whether it's `IN_SCOPE`, `PARTIALLY_IN_SCOPE`, `OUT_OF_SCOPE`, or too vague to judge
   (`NEEDS_CLARIFICATION`), and explains why.
3. Logs it, and recalculates the running totals: extra hours, percentage over the original estimate,
   how many requests have been out of scope, and how many genuinely **new kinds of functionality**
   have been introduced.
4. Watches the *cumulative* picture, not just the latest request - because five small "reasonable"
   asks can add up to a project that no longer resembles what was agreed, even if none of them looked
   alarming by itself.
5. Warns the freelancer once that cumulative drift crosses a threshold, and recommends re-negotiating
   scope.

## 3. Why this needs an AI model, not just Python rules

A professor's fair question: *"why not just write if/else rules?"*

Deterministic Python is genuinely the right tool for some of this - adding up hours, calculating a
percentage, counting requests. Those are done in plain Python in this project (see `src/drift.py`),
**not** left to the model, because arithmetic should never be a guess.

But deciding *whether* "add a loyalty points system" fits inside "a 5-page marketing website" is a
language-understanding problem. A keyword match cannot tell that authentication, loyalty points, and
an admin dashboard are all instances of the same underlying thing: the project turning from a
marketing site into an operations platform. That requires actually understanding what each request
*means* and what it implies - which is exactly the part handed to the model here, and only that part.

## 4. Architecture

```
freelancer describes original scope  ->  stored once as structured project state (plain Python, no model call)

client request (natural language)
        |
        v
   ScopeKeeperAgent  --------------------->  language model (Groq)
        |                                     - reads the request + a short scope summary
        |                                     - decides classification, affected deliverables,
        |                                       new capabilities, an hours estimate, reasoning
        |
        v
   log_request(...)   <- tool call, arguments produced by the model
        |
        v
   validation.py checks the arguments are well-formed and in range
        |
        v
   drift.py recalculates cumulative hours / out-of-scope ratio / drift score  (pure Python)
        |
        v
   get_scope_status()  <- tool call, lets the model see the updated cumulative picture
        |
        v
   model reads the numbers and explains the situation / recommends next steps
```

The model never touches the arithmetic, and the Python layer never guesses what a sentence means -
each side does the part it's actually reliable at.

## 5. The agent loop

This is a genuine multi-step loop, not a fixed sequence: the number of tool calls the model makes,
and what it does next, depends on what the tool results say - not on a hardcoded script.

```
user message
   -> model reads project state + message
   -> model decides: does this need a tool?
        - new client request  -> call log_request(...) with its own classification
        - "are we drifting?"  -> call get_scope_status()
        - neither             -> just reply
   -> tool runs, returns real, freshly-computed data
   -> model sees the tool result and decides: answer now, or take one more step?
   -> repeat, up to 4 steps
```

Safety rails: a hard cap of 4 steps per turn, and an immediate stop if the model tries to repeat the
exact same tool call with the exact same arguments twice in a row (see `_complete_with_retry` /
the `last_call_signature` check in `src/agent.py`).

In [ ]:
import sys, os

# Make the src/ package importable from the notebook, wherever it's opened from.
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

# Windows terminals default to a limited encoding that can't print some
# characters the model uses (smart quotes, em-dashes, checkmarks). Jupyter
# itself doesn't need this, but it's harmless and avoids a crash if any
# cell's output is ever piped through a plain Windows console.
if sys.platform == "win32":
    sys.stdout.reconfigure(encoding="utf-8")

# If running in Colab, uncomment the next line to install dependencies:
# !pip install -q openai python-dotenv

print("Ready.")

## 6. API key setup (secure)

The key is never hardcoded in this notebook or committed to the repository. It's read from an
environment variable, loaded from a local `.env` file (see `.env.example`) if one exists, or - if
neither is set - asked for securely at runtime so it never appears in the notebook's saved output.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

if not (os.getenv("GROQ_API_KEY") or os.getenv("GITHUB_TOKEN")):
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY (input hidden): ")

print("API key configured:", bool(os.getenv("GROQ_API_KEY") or os.getenv("GITHUB_TOKEN")))

## 7. Model configuration, and why

| Setting | Value | Why |
|---|---|---|
| Model | `openai/gpt-oss-120b` on Groq | Reliable at correctly using tools, generous free tier, no reason to add a second provider for a course project. |
| Fallback model | `llama-3.1-8b-instant` | Only used if the main model call fails twice in a row - keeps the demo running instead of stopping cold. |
| Temperature | `0.2` | This is a judgement task (classify, don't write creatively), so it should be low - but not 0, so a retry can genuinely differ from a failed first attempt instead of repeating it. |
| Max output tokens | `600` | A classification + short reasoning is naturally short; this keeps answers focused and bounds cost per call. |
| Max loop iterations | `4` | Enough for a real multi-step exchange; beyond that, stopping and asking for more detail is the honest response, not a longer loop. |

**Token strategy:** each call sends the system instructions, a short plain-English summary of the
project (`ProjectState.compact_context()` - a handful of lines), and only the last few conversation
turns. The full request history is *not* resent on every call - the model only sees it when it
explicitly calls `get_scope_status()`, which is exactly when it actually needs it. This keeps each
call small regardless of how many requests have been logged so far.

In [ ]:
from agent import ScopeKeeperAgent

agent = ScopeKeeperAgent()
print(f"Agent ready. Using model: {agent.model} (fallback: {agent.fallback_model})")

## 8. The two tools

**`log_request(description, hours_estimate, classification, affected_deliverables, new_capabilities, reasoning, confidence)`**
Called by the model for every new client request. The *classification itself* is produced by the
model's reasoning (this is the tool-call arguments, i.e. structured output) - the tool's own Python
code only validates those arguments (see `src/validation.py`) and, once they pass, stores the record
and recalculates the cumulative drift numbers deterministically (`src/drift.py`).

**`get_scope_status()`**
Returns the original scope, the full request history, and the current cumulative numbers (extra
hours, percentage increase, out-of-scope count, distinct new capability groups, drift score and
level). The model calls this when reasoning about the overall picture rather than a single request.

Setting up the *original* scope is deliberately **not** a tool - there's no judgement call involved in
typing in the agreed deliverables, so it's a plain Python method (`agent.start_project(...)`), keeping
the two tools reserved for the steps that actually require the model's reasoning.

## 9. Memory

Memory here is a structured `ProjectState` object (`src/memory.py`), not "the whole conversation
pasted back to the model." It holds the original deal, the full list of logged requests, and a short
rolling conversation log used only for natural back-and-forth phrasing. The facts the model actually
reasons over - deliverables, exclusions, cumulative totals - always come from this structured state,
so a stray line in the chat history can't quietly change what the model believes the deal was.

## 10. Live demo

A realistic scenario: a ₹80,000, 20-hour restaurant marketing website. Watch the classification for
each request, and how the cumulative picture changes as small, individually-reasonable requests add up.

In [ ]:
agent = ScopeKeeperAgent()
agent.start_project(
    name="Restaurant marketing website",
    objective="A marketing website for a restaurant to attract new customers",
    budget=80000,
    estimated_hours=20,
    deliverables=["Home page", "About page", "Menu page", "Contact page", "Gallery page"],
    exclusions=["online ordering", "user accounts", "payment processing", "staff or admin tools"],
)

def show_trace(agent):
    print("--- STEPS TAKEN ---")
    for event in agent.trace:
        print(event)
    print()

requests = [
    "Can we change the contact page so customers can submit a message directly?",
    "Can customers create an account and log in?",
    "Can customers save their favorite menu items?",
    "Can the restaurant staff have an admin dashboard to manage orders?",
]

for r in requests:
    print(f"CLIENT REQUEST: {r}\n")
    answer = agent.run(f"New client request: {r}")
    print(f"AGENT: {answer}\n")
    show_trace(agent)
    print("=" * 70)

In [ ]:
# The "wow" question - ask it directly and let the agent read back the cumulative state.
answer = agent.run("Are we drifting from the original agreement? What should I do?")
print(answer)
show_trace(agent)

In [ ]:
import json
print(json.dumps(agent.tools.get_scope_status()["cumulative"], indent=2))

## 11. Test suite

The deterministic parts (drift math, argument validation, the tools themselves) are tested without
any API call - they're plain Python, so they're checked as plain Python.

In [ ]:
import subprocess
for test_file in ["tests/test_drift.py", "tests/test_validation.py", "tests/test_tools.py"]:
    result = subprocess.run([sys.executable, test_file], capture_output=True, text=True)
    print(f"$ python {test_file}")
    print(result.stdout.strip() or result.stderr.strip())
    print()

## 12. Accuracy evaluation

This is the part that genuinely needs the model, because it's testing whether its judgement is good,
not just whether the code runs. 32 hand-labelled single-request cases (`data/evaluation_cases.json`,
covering all four classification labels with varied, non-templated wording) and 4 hand-labelled
cumulative sequences (`data/drift_sequences.json`). If no API key is set, this honestly reports
"not measured" instead of a made-up number - run this cell with a real key to get real numbers.

In [ ]:
sys.path.insert(0, os.path.join(os.getcwd(), "tests"))
from evaluation import run_classification_eval, run_drift_eval

if not (os.getenv("GROQ_API_KEY") or os.getenv("GITHUB_TOKEN")):
    print("No API key set - accuracy not measured. Set GROQ_API_KEY above and re-run this cell.")
else:
    print("Single-request classification accuracy:")
    c = run_classification_eval()
    print(f"  {c['accuracy']*100:.1f}% ({c['correct']}/{c['total']})")
    for label, stats in c["per_class"].items():
        print(f"  {label}: precision={stats['precision']}, recall={stats['recall']}, support={stats['support']}")

    print("\nCumulative drift-sequence accuracy:")
    d = run_drift_eval()
    print(f"  {d['accuracy']*100:.1f}% ({d['correct']}/{d['total']})")
    for r in d["results"]:
        status = "OK" if r["match"] else "MISMATCH"
        print(f"  [{status}] {r['id']}: expected={r['expected_final_drift_level']}, actual={r['actual_drift_level']}")

## 13. Failure handling tests

Deliberately breaking things, to show they're handled rather than crashing.

In [ ]:
# Malformed tool arguments (this is what happens if the model ever returns something invalid)
from tools import ScopeTools
from memory import ProjectState

state = ProjectState()
state.name, state.estimated_hours = "Test project", 10
broken_tools = ScopeTools(state)

result = broken_tools.log_request(
    description="Add a form",
    hours_estimate=-999,          # invalid: negative
    classification="MAYBE",       # invalid: not one of the four allowed values
    reasoning="test",
    confidence="HIGH",
)
print("Malformed input result:", result)
assert result["status"] == "error"
print("Handled without crashing, and nothing was stored.")

In [ ]:
# Duplicate request: the same request logged twice should not double-count.
state2 = ProjectState()
state2.name, state2.estimated_hours = "Test project", 10
dup_tools = ScopeTools(state2)

dup_tools.log_request(description="Add customer login", hours_estimate=5, classification="OUT_OF_SCOPE",
                       reasoning="Adds authentication.", confidence="HIGH", new_capabilities=["authentication"])
second = dup_tools.log_request(description="Add customer login", hours_estimate=5, classification="OUT_OF_SCOPE",
                                reasoning="Adds authentication.", confidence="HIGH", new_capabilities=["authentication"])
print("Second identical request result:", second["status"])
assert second["status"] == "duplicate"
assert len(state2.requests) == 1
print("Not double-counted.")

In [ ]:
# Loop-step limit: force max_iterations=1 on a request that would normally take two tool calls,
# and confirm the agent stops gracefully instead of hanging or crashing.
if os.getenv("GROQ_API_KEY") or os.getenv("GITHUB_TOKEN"):
    limited_agent = ScopeKeeperAgent()
    limited_agent.start_project(
        name="Restaurant marketing website", objective="Marketing site", budget=80000, estimated_hours=20,
        deliverables=["Home page", "Contact page"], exclusions=["online ordering"],
    )
    answer = limited_agent.run("New client request: add an online ordering system.", max_iterations=1)
    print(answer)
else:
    print("Skipped - needs an API key to demonstrate live (the cap itself is unit-tested in code review).")

## 14. Results

Measured against the real Groq API on 2026-08-30 (`tests/evaluation.py`), across five runs - all
reported, not just the best one.

**Classification accuracy:** 90.6% -> 84.4% -> 84.4% -> 6.2% (discarded, a bug not a real result) -> **96.9% (31/32)**
on the final clean run. The one remaining miss is an arguably borderline label (an FAQ-section request).

**What happened between run 3 and run 5, briefly:**
1. Runs 1-2 showed the model answering confidently on vague requests instead of using
   `NEEDS_CLARIFICATION`. Fixed with a clearer system-prompt instruction (run 3): helped that specific
   metric, but a new pattern (PARTIALLY_IN_SCOPE calls slipping to IN_SCOPE) showed up alongside it.
2. Run 4 (after adding worked boundary examples to the prompt) came back at 6.2% - almost every
   prediction was `NO_CALL`. Investigating a single request directly showed this wasn't a prompt
   problem at all: the free-tier rate limit had been hit after ~130 calls in one session, the retry
   logic fell back to a second model, and that fallback model (`llama-3.1-8b-instant`) no longer
   exists on Groq - so every retry failed with a 404, silently producing "AI unavailable" instead of a
   classification. **Fixed** by switching the fallback to a model that's actually available
   (`openai/gpt-oss-20b`) and lengthening the rate-limit backoff from 1.5s to 8s.
3. Run 5, right after the fix: 96.9% (31/32) - the real effect of the boundary-examples prompt change,
   once the infrastructure bug wasn't drowning it out.

**Cumulative drift-sequence accuracy: 3/4 (75%)** on the final clean run (down from an earlier 4/4).
The miss was traced to the model's own hours estimate for one request varying between runs - since
hours make up 40% of the drift score formula, that alone was enough to push a borderline sequence from
MODERATE to HIGH. Not a classification error; a genuine limitation of scoring on a single run (see
README, "Measured accuracy", for the full trace of the reasoning).

**Unit tests:** all passing throughout (they don't depend on the model, so none of the above affected
them) - `test_drift.py`, `test_validation.py`, `test_tools.py`.

Full detail, per-case misses, and the exact bug diagnosis: README, "Measured accuracy".

## 15. Limitations

- Only evaluated on one project scenario (a restaurant marketing website) - accuracy on very different
  domains (e.g. software consulting, design work) is untested.
- `NEEDS_CLARIFICATION` requests don't currently feed into the drift score at all - a project that
  gets a lot of vague requests wouldn't show any elevated drift, even though vagueness is itself a
  minor warning sign. Left out deliberately rather than guessing a weight for it.
- Duplicate detection is exact-match only (after trimming whitespace/case) - a client rephrasing the
  same request slightly would be logged as a new item.
- Single project per agent instance - no persistence to disk between notebook runs (deliberate, see
  README, but worth stating plainly here too).

## 16. Future improvements

- Run the classification evaluation several times and average the results - single before/after runs
  aren't enough to confidently separate a real prompt-tuning effect from normal model variance (see
  Section 14).
- Sharpen the IN_SCOPE / PARTIALLY_IN_SCOPE boundary in the system prompt with a couple of worked
  examples - this is the single biggest source of measured misclassifications.
- A lightweight "ambiguity score" that factors `NEEDS_CLARIFICATION` frequency into the drift picture.
- Optional: a third, non-decision-making helper that drafts a polite change-order message once drift
  crosses a threshold (explicitly left out of the core build so it doesn't compete with the two tools
  that matter for the rubric).
- Persisting project state to a small local file so a project can be picked back up in a later
  session, not just within one notebook run.